In [ ]:
# ── KAGGLE SETUP (only runs on Kaggle, no-op everywhere else) ───────────────
import os, sys
from pathlib import Path

if os.environ.get("KAGGLE_KERNEL_RUN_TYPE") or Path("/kaggle/input").exists():
    import subprocess

    REPO_URL = "https://github.com/ltruonghai257/fake-new-detection.git"
    CLONE_DIR = Path("/kaggle/working/fake-news-detection")
    if not CLONE_DIR.exists():
        subprocess.run(["git", "clone", REPO_URL, str(CLONE_DIR)], check=True)
        print(f"Cloned → {CLONE_DIR}")
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "-r",
            str(CLONE_DIR / "requirements.txt"),
        ],
        check=True,
    )

    def _find_hdf5_root():
        for d in Path("/kaggle/input").rglob("processed_data/hdf5"):
            if d.is_dir():
                return d.parent.parent
        return None

    hdf5_root = _find_hdf5_root()
    if hdf5_root is None:
        raise RuntimeError("processed_data/hdf5/ not found under /kaggle/input/")
    (CLONE_DIR / ".env.kaggle").write_text(f"DATA_ROOT={hdf5_root}\nPLATFORM=kaggle\n")
    print(f"DATA_ROOT={hdf5_root}")
else:
    print("Not Kaggle — setup skipped.")

In [ ]:
# ── Environment + path setup ────────────────────────────────────────────────
import os, sys
from pathlib import Path


def _detect_platform():
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE") or Path("/kaggle/input").exists():
        return "kaggle"
    try:
        import google.colab

        return "colab"
    except ImportError:
        pass
    if Path("/workspace").exists():
        return "vastai"
    if sys.platform == "darwin":
        return "mac"
    return "windows"


PLATFORM = _detect_platform()

try:
    _nb_path = Path(__file__).resolve()
except NameError:
    _nb_path = Path.cwd()

if PLATFORM == "colab":
    from google.colab import drive

    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/drive/MyDrive/Thesis_Final/fake-news-detection")
elif PLATFORM == "kaggle":
    PROJECT_ROOT = Path(
        os.environ.get("KAGGLE_PROJECT_ROOT", "/kaggle/working/fake-news-detection")
    )
else:
    PROJECT_ROOT = (
        _nb_path.parents[1]
        if _nb_path.name.endswith(".ipynb")
        else Path.cwd().parents[1]
    )

sys.path.insert(0, str(PROJECT_ROOT))

from dotenv import load_dotenv

_env_file = PROJECT_ROOT / f".env.{PLATFORM}"
if not _env_file.exists():
    _env_file = PROJECT_ROOT / ".env"
load_dotenv(_env_file, override=True)

from src.utils.env_utils import get_data_root

DATA_ROOT = get_data_root()
print(f"Platform : {PLATFORM}")
print(f"DATA_ROOT: {DATA_ROOT}  (exists={DATA_ROOT.exists()})")

# EANN — Event Adversarial Neural Network for Fake News Detection

> **This notebook trains the EANN model only.** It is a fully independent architecture
> from COOLANT (`src/models/coolant_official.py`) and from CAFE
> (`training-fake-news-with-CAFE.ipynb`) — no model class from either is imported or
> used anywhere below. The only thing shared with those pipelines is the **raw
> pre-extracted feature files** (PhoBERT text embeddings + ResNet50 image embeddings,
> stored as `*.h5`) and the generic HDF5 loading utility that reads them — these are
> just feature caches, not part of any model.

Paper: _EANN: Event Adversarial Neural Networks for Multi-Modal Fake News Detection_ (KDD 2018)  
Original repo: https://github.com/yaqingwang/EANN-KDD18

**Main idea:** a _event discriminator_ is trained adversarially (via a Gradient
Reversal Layer, GRL) against the shared text+image representation, so the fused
features become **event-invariant** — this helps the detector generalize to fake
news about newly emerged events it has never seen at train time.

**Training scheme (two heads sharing one fused representation, joint loss):**

```
fused = concat(TextCNN(text), ImageFC(image))                       (B, 2*hidden_dim)

Head 1 — Detector:   class_logits  = ClassClassifier(fused)
                     L_CLASS = CrossEntropyLoss(class_logits, det_label)
                     Real = (text_i,   image_i)            → det_label 0
                     Fake = (text_i,   image_i+shift)      → det_label 1

Head 2 — Adversary:  domain_logits = DomainClassifier(GRL(fused, λ))
                     L_DOMAIN = CrossEntropyLoss(domain_logits, event_label)

Total:  L = L_CLASS + L_DOMAIN   (single Adam optimizer, single backward pass)
        GRL flips the sign of the domain-loss gradient flowing into `fused`,
        so the shared encoder is pushed to make `fused` event-*indistinguishable*
        while the detector head still learns to separate real/fake.
```

**Model classes used (all defined inline in this notebook, self-contained):**
`GradReverse`, `TextCNN_EANN`, `EANNModel`.

**Architecture adaptations vs original paper:**

-   Text branch: original EANN looks up word2vec embeddings via `nn.Embedding` then runs
    `Conv2d` windows over them. Our data already contains pre-extracted **PhoBERT**
    features `(B, 768, seq_len)`, so `TextCNN_EANN` applies the same multi-kernel
    (`window_size=(1,2,3,4)`, `filter_num=20`) max-pool-and-concat design directly with
    `Conv1d` over the 768 channels — no embedding lookup needed.
-   Image branch: original runs a frozen **VGG19** backbone then `Linear(4096→hidden)`.
    Our data already contains pre-extracted **ResNet50** features `(B, 2048)`, so we
    use `Linear(2048→hidden)` directly.
-   `class_classifier` / `domain_classifier` return **raw logits** (no `Softmax` layer).
    The original code applies `nn.Softmax` inside both classifiers _and_ then feeds the
    result into `nn.CrossEntropyLoss` (which itself applies `log_softmax`) — a redundant
    double-softmax in the original repo. Removed here for numerical correctness; behavior
    is otherwise unchanged.
-   **Event labels**: the original Weibo/Twitter datasets ship with hand-clustered event
    IDs per post. Our HDF5 files have no such topic/event metadata, so we derive
    **pseudo-event labels** via K-Means clustering on mean-pooled PhoBERT text vectors
    (fit once on the train split, see the dedicated cell below). This preserves EANN's
    core mechanism (adversarial event-invariance) without requiring hand-annotated
    event metadata.
-   Unused branches from the original `CNN_Fusion` (`lstm`, `text_encoder`,
    `attention_layer`, `social` context) are **not ported** — they are defined in the
    original `__init__` but never referenced in its `forward()`, i.e. dead code there.

**Checkpoints/outputs** are written to `checkpoints_eann/` (run name prefix
`eann_...`, Kaggle dataset slug `eann-best`) — fully separate from COOLANT's
`checkpoints_coolant_v2/` and CAFE's `checkpoints_cafe/`.


In [ ]:
# ── CONFIG — edit this cell only ────────────────────────────────────────────
CONFIG = {
    "paths": {
        # Pre-extracted PhoBERT+ResNet50 feature caches (shared raw data, not a model).
        "train_hdf5": DATA_ROOT / "processed_data" / "hdf5" / "coolant_train.h5",
        "dev_hdf5": DATA_ROOT / "processed_data" / "hdf5" / "coolant_dev.h5",
        "test_hdf5": DATA_ROOT / "processed_data" / "hdf5" / "coolant_test.h5",
        # EANN-specific checkpoint output — separate from COOLANT/CAFE dirs.
        "checkpoint_root": DATA_ROOT / "training" / "checkpoints_eann",
    },
    "model": {
        # Feature dims (must match HDF5 preprocessing output)
        "text_in_channels": 768,  # PhoBERT embed dim
        "image_input_dim": 2048,  # ResNet50 feature dim
        # EANN architecture (original paper defaults)
        "cnn_filter_num": 20,
        "cnn_window_size": (1, 2, 3, 4),
        "hidden_dim": 64,
        "class_num": 2,  # real (0) / fake (1)
        "event_num": 10,  # pseudo-event clusters (K-Means, see below)
        "grl_lambda": 1.0,  # gradient reversal strength
    },
    "training": {
        "batch_size": 64,
        "max_epochs": 100,
        "negative_shift": 3,
        "min_batch_size": 4,
        "grad_clip": 1.0,
        "lr": 1e-3,
        "lr_decay_rate": 10,  # lr(p) = lr / (1 + lr_decay_rate * p) ** lr_decay_power
        "lr_decay_power": 0.75,  # p = epoch / max_epochs   (matches original EANN.py schedule)
        "weight_decay": 0.0,
        "seed": 42,
        "early_stopping_patience": 15,  # stop if val_macro_f1 doesn't improve for N epochs
    },
    "pseudo_event": {
        "n_clusters": 10,  # must match model.event_num
        "kmeans_seed": 42,
        "kmeans_n_init": 10,
    },
    "checkpointing": {
        "selection_metric": "val_macro_f1",
        "checkpoint_every": 5,
    },
    "safety": {
        "smoke_test": False,
        "smoke_batches": 2,
        "resume_from_checkpoint": None,
    },
}

# Kaggle: redirect writable outputs
if os.environ.get("KAGGLE_KERNEL_RUN_TYPE") or Path("/kaggle/input").exists():
    CONFIG["paths"]["checkpoint_root"] = Path("/kaggle/working/checkpoints_eann")

CONFIG["paths"]["checkpoint_root"].mkdir(parents=True, exist_ok=True)
print("CONFIG loaded — training EANN model (independent from COOLANT and CAFE).")
print(f"Checkpoints will be saved under: {CONFIG['paths']['checkpoint_root']}")

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import gc, json, random, math, shutil
from datetime import datetime

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
from sklearn.cluster import KMeans
from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    confusion_matrix,
    roc_auc_score,
)

# Generic raw-feature HDF5 loader (PhoBERT + ResNet50 caches, NOT the COOLANT model).
# This utility only reads pre-extracted (caption, image) tensors from disk; it has no
# dependency on the COOLANT architecture and is reused here purely for I/O convenience.
from src.preprocessing.coolant.pair_dataset import (
    create_coolant_dataloaders as create_feature_dataloaders,
)

print(f"PyTorch: {torch.__version__}")

In [ ]:
# ── EANN Model Definition (adapted for PhoBERT + ResNet50 features) ──────────
#
# Faithful to the original EANN code (yaqingwang/EANN-KDD18, src/EANN.py), with
# targeted adaptations (see markdown cell above for the full rationale):
#   1. TextCNN_EANN: Conv1d over 768-dim PhoBERT features (no Embedding lookup)
#   2. Image branch: Linear(2048→hidden) instead of frozen VGG19 + Linear(4096→hidden)
#   3. Classifiers output raw logits (no redundant Softmax before CrossEntropyLoss)
# ─────────────────────────────────────────────────────────────────────────────


class GradReverse(torch.autograd.Function):
    """Gradient Reversal Layer (Ganin & Lempitsky, 2015) — used by EANN's
    domain/event adversary to push the shared encoder toward event-invariant
    features. Forward is identity; backward flips (and scales) the gradient.
    """

    @staticmethod
    def forward(ctx, x, lambd):
        ctx.lambd = lambd
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):
        return -ctx.lambd * grad_output, None


def grad_reverse(x, lambd):
    return GradReverse.apply(x, lambd)


class TextCNN_EANN(nn.Module):
    """CNN text encoder adapted for PhoBERT features (B, 768, seq_len).

    Original EANN embeds word IDs via nn.Embedding then runs Conv2d windows over
    (1, seq_len, emb_dim). Our data is already (B, 768, seq_len) — apply Conv1d
    directly over the 768 channels, same multi-kernel max-pool-concat design.
    """

    def __init__(
        self, in_channels=768, filter_num=20, window_size=(1, 2, 3, 4), hidden_dim=64
    ):
        super().__init__()
        self.convs = nn.ModuleList(
            [nn.Conv1d(in_channels, filter_num, kernel_size=k) for k in window_size]
        )
        self.fc1 = nn.Linear(filter_num * len(window_size), hidden_dim)

    def forward(self, x):
        # x: (B, 768, seq_len) — already (B, C, L), no permute needed
        feats = [
            F.leaky_relu(conv(x)) for conv in self.convs
        ]  # [(B, filter_num, L'), ...]
        feats = [
            F.max_pool1d(f, f.size(2)).squeeze(2) for f in feats
        ]  # [(B, filter_num), ...]
        feats = torch.cat(feats, dim=1)  # (B, filter_num*len(window_size))
        return F.leaky_relu(self.fc1(feats))  # (B, hidden_dim)


class EANNModel(nn.Module):
    """Adapted CNN_Fusion: shared text+image encoder, detector head, adversarial
    event-discriminator head (via GradReverse).
    """

    def __init__(
        self,
        text_in_channels=768,
        image_input_dim=2048,
        cnn_filter_num=20,
        cnn_window_size=(1, 2, 3, 4),
        hidden_dim=64,
        class_num=2,
        event_num=10,
    ):
        super().__init__()
        self.text_cnn = TextCNN_EANN(
            in_channels=text_in_channels,
            filter_num=cnn_filter_num,
            window_size=cnn_window_size,
            hidden_dim=hidden_dim,
        )
        self.image_fc1 = nn.Linear(image_input_dim, hidden_dim)

        # Head 1: fake/real detector
        self.class_classifier = nn.Linear(2 * hidden_dim, class_num)

        # Head 2: adversarial event/domain discriminator (trained via GRL)
        self.domain_classifier = nn.Sequential(
            nn.Linear(2 * hidden_dim, hidden_dim),
            nn.LeakyReLU(True),
            nn.Linear(hidden_dim, event_num),
        )

    def forward(self, text, image, grl_lambda=1.0):
        text_feat = self.text_cnn(text)  # (B, hidden_dim)
        image_feat = F.leaky_relu(self.image_fc1(image))  # (B, hidden_dim)
        fused = torch.cat([text_feat, image_feat], dim=1)  # (B, 2*hidden_dim)

        class_output = self.class_classifier(fused)  # (B, class_num)
        reversed_feat = grad_reverse(fused, grl_lambda)
        domain_output = self.domain_classifier(reversed_feat)  # (B, event_num)
        return class_output, domain_output


print("EANN model classes defined.")

In [ ]:
# ── Device, seed, dataloaders (raw features only — EANN model built later) ───
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available() else "cpu"
)
seed_everything(CONFIG["training"]["seed"])
print(f"Device: {DEVICE}")

loaders, datasets = create_feature_dataloaders(
    str(CONFIG["paths"]["train_hdf5"]),
    str(CONFIG["paths"]["dev_hdf5"]),
    str(CONFIG["paths"]["test_hdf5"]),
    batch_size=CONFIG["training"]["batch_size"],
)

if CONFIG["safety"]["smoke_test"]:
    from itertools import islice

    class _SmokeLoader:
        def __init__(self, l, n):
            self._l, self._n = l, n

        def __iter__(self):
            return islice(self._l, self._n)

        def __len__(self):
            return min(self._n, len(self._l))

    loaders = {
        k: _SmokeLoader(v, CONFIG["safety"]["smoke_batches"])
        for k, v in loaders.items()
    }
    print(f"SMOKE TEST: {CONFIG['safety']['smoke_batches']} batches/split")

# Verify shapes
_cap, _img, _ = next(iter(loaders["train"]))
print(f"caption: {tuple(_cap.shape)}  (B, embed=768, seq=128)  [PhoBERT features]")
print(f"image:   {tuple(_img.shape)}  (B, 2048)  [ResNet50 features]")
print("These are raw feature tensors — the EANN model is built and trained below.")

In [ ]:
# ── Fit pseudo-event clusters (K-Means) on mean-pooled train text features ──
#
# EANN's adversarial head needs an "event" label per sample. The original Weibo/
# Twitter datasets ship with hand-annotated event IDs; our HDF5 files don't carry
# any topic/event metadata. We derive a pseudo-event label by clustering mean-
# pooled PhoBERT text vectors with K-Means — this recovers the same *purpose*
# (grouping semantically-similar news so the adversary has something non-trivial
# to discriminate) without requiring hand-annotated labels.

pe_cfg = CONFIG["pseudo_event"]
assert (
    pe_cfg["n_clusters"] == CONFIG["model"]["event_num"]
), "pseudo_event.n_clusters must match model.event_num"

print(
    f"Collecting mean-pooled text features from train split for K-Means (k={pe_cfg['n_clusters']})..."
)
_pooled_feats = []
for caption, _, _ in tqdm(loaders["train"], desc="Pooling text features"):
    _pooled_feats.append(
        caption.mean(dim=2).cpu().numpy()
    )  # (B, 768) mean over seq_len
    # .cpu() is required: CoolantPairDataset already moves tensors to DEVICE
    # (see src/preprocessing/coolant/pair_dataset.py), so `caption` may be a
    # CUDA tensor here even before we call .to(device) ourselves later.
_pooled_feats = np.concatenate(_pooled_feats, axis=0)
print(f"Collected {_pooled_feats.shape[0]} vectors, dim={_pooled_feats.shape[1]}")

event_kmeans = KMeans(
    n_clusters=pe_cfg["n_clusters"],
    random_state=pe_cfg["kmeans_seed"],
    n_init=pe_cfg["kmeans_n_init"],
)
event_kmeans.fit(_pooled_feats)
print(
    f"K-Means fit done. Cluster sizes: {np.bincount(event_kmeans.labels_, minlength=pe_cfg['n_clusters'])}"
)
del _pooled_feats


def get_pseudo_event_labels(text_batch):
    """text_batch: (B, 768, seq_len) -> mean-pool over seq -> KMeans.predict -> (B,) long tensor."""
    pooled = text_batch.mean(dim=2).cpu().numpy()
    labels = event_kmeans.predict(pooled)
    return torch.from_numpy(labels).long()

In [ ]:
# ── Build EANN model, optimizer, LR schedule ─────────────────────────────────
m_cfg = CONFIG["model"]

model = EANNModel(
    text_in_channels=m_cfg["text_in_channels"],
    image_input_dim=m_cfg["image_input_dim"],
    cnn_filter_num=m_cfg["cnn_filter_num"],
    cnn_window_size=m_cfg["cnn_window_size"],
    hidden_dim=m_cfg["hidden_dim"],
    class_num=m_cfg["class_num"],
    event_num=m_cfg["event_num"],
).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters())
print(f"EANNModel parameters: {n_params:,}")

loss_ce = nn.CrossEntropyLoss()

lr = CONFIG["training"]["lr"]
wd = CONFIG["training"]["weight_decay"]
optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()), lr=lr, weight_decay=wd
)


def _lr_lambda(epoch):
    # Matches original EANN.py: lr(p) = lr / (1 + decay_rate * p) ** decay_power
    p = epoch / max(1, CONFIG["training"]["max_epochs"])
    rate = CONFIG["training"]["lr_decay_rate"]
    power = CONFIG["training"]["lr_decay_power"]
    return 1.0 / (1.0 + rate * p) ** power


scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, _lr_lambda)
GRL_LAMBDA = m_cfg["grl_lambda"]

print(f"Optimizer: Adam (lr={lr}, wd={wd})")
print(
    f"LR schedule: lr / (1 + {CONFIG['training']['lr_decay_rate']}*p)**{CONFIG['training']['lr_decay_power']}"
)
print(f"GRL lambda (fixed): {GRL_LAMBDA}")

In [ ]:
# ── Pair construction helper + sanity check ──────────────────────────────────


def make_detection_batch(text, image, shift):
    """Create detection batch: real (matched) + fake (unmatched).

    Real  = original (text_i, image_i)          → label 0
    Fake  = mismatched (text_i, image_{i+shift}) → label 1

    Returns:
        det_text:   (2B, 768, 128)
        det_image:  (2B, 2048)
        det_labels: (2B,) long
    """
    B = text.size(0)
    det_text = torch.cat([text, text], dim=0)
    det_image = torch.cat([image, image.roll(shift, dims=0)], dim=0)
    det_labels = torch.cat(
        [
            torch.zeros(B, dtype=torch.long, device=text.device),  # 0 = real
            torch.ones(B, dtype=torch.long, device=text.device),  # 1 = fake
        ],
        dim=0,
    )
    return det_text, det_image, det_labels


# ── Sanity check (one batch) ─────────────────────────────────────────────────
model.train()
_cap, _img, _ = next(iter(loaders["train"]))
_cap, _img = _cap.to(DEVICE), _img.to(DEVICE)

_dc, _di, _dl = make_detection_batch(_cap, _img, CONFIG["training"]["negative_shift"])
_de = get_pseudo_event_labels(_dc).to(DEVICE)

_class_out, _domain_out = model(_dc, _di, GRL_LAMBDA)
_l_class = loss_ce(_class_out, _dl)
_l_domain = loss_ce(_domain_out, _de)

print(f"class_logits  shape: {_class_out.shape}   (2B, class_num={m_cfg['class_num']})")
print(f"domain_logits shape: {_domain_out.shape}  (2B, event_num={m_cfg['event_num']})")
print(
    f"L_CLASS  = {_l_class.item():.4f}   {'OK' if torch.isfinite(_l_class) else 'NaN!'}"
)
print(
    f"L_DOMAIN = {_l_domain.item():.4f}   {'OK' if torch.isfinite(_l_domain) else 'NaN!'}"
)
print("Sanity check passed.")

In [ ]:
# ── Run directory + checkpoint helpers ──────────────────────────────────────
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
run_name = f"eann_{timestamp}"
run_dir = CONFIG["paths"]["checkpoint_root"] / run_name
run_dir.mkdir(parents=True, exist_ok=True)
print(f"Run dir: {run_dir}")


def save_checkpoint(path, epoch, history, metrics):
    torch.save(
        {
            "model_state": model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "epoch": epoch,
            "metrics": metrics,
            "training_history": history,
            "config": CONFIG,
        },
        path,
    )


def load_checkpoint(path):
    ckpt = torch.load(path, map_location=DEVICE)
    model.load_state_dict(ckpt["model_state"])
    print(f"Loaded checkpoint from epoch {ckpt['epoch']}: {path}")
    return ckpt


start_epoch = 0
if CONFIG["safety"]["resume_from_checkpoint"]:
    _ckpt = load_checkpoint(CONFIG["safety"]["resume_from_checkpoint"])
    start_epoch = _ckpt["epoch"] + 1

# ── Kaggle auto-upload best model ────────────────────────────────────────────
IS_KAGGLE = bool(
    os.environ.get("KAGGLE_KERNEL_RUN_TYPE") or Path("/kaggle/input").exists()
)
_KAGGLE_API = None
_KAGGLE_UPLOAD_OK = False

if IS_KAGGLE:
    try:
        from kaggle_secrets import UserSecretsClient

        _sec = UserSecretsClient()
        os.environ["KAGGLE_USERNAME"] = _sec.get_secret("KAGGLE_USERNAME")
        os.environ["KAGGLE_KEY"] = _sec.get_secret("KAGGLE_KEY")
        from kaggle.api.kaggle_api_extended import KaggleApi

        _KAGGLE_API = KaggleApi()
        _KAGGLE_API.authenticate()
        _KAGGLE_UPLOAD_OK = True
        print(f"Kaggle API authenticated as {os.environ['KAGGLE_USERNAME']}")
    except Exception as e:
        print(
            f"Kaggle secrets/API not set up ({e}) — best model saved to Output tab only."
        )

_BEST_DS_SLUG = "eann-best"


def kaggle_upload_best(ckpt_path, epoch, f1):
    if not _KAGGLE_UPLOAD_OK:
        return
    try:
        staging = Path("/kaggle/working/_ds_best_eann")
        staging.mkdir(exist_ok=True)
        for old in staging.glob("*.pth"):
            old.unlink()
        shutil.copy2(ckpt_path, staging / "best_macro_f1.pth")
        (staging / "best_info.json").write_text(
            json.dumps(
                {
                    "run": run_name,
                    "epoch": int(epoch),
                    "val_macro_f1": float(f1),
                    "saved_at": datetime.now().isoformat(),
                },
                indent=2,
            )
        )
        (staging / "dataset-metadata.json").write_text(
            json.dumps(
                {
                    "title": "EANN Best Model",
                    "id": f"{os.environ['KAGGLE_USERNAME']}/{_BEST_DS_SLUG}",
                    "licenses": [{"name": "other"}],
                }
            )
        )
        try:
            _KAGGLE_API.dataset_create_version(
                str(staging),
                f"ep{epoch} f1={f1:.4f} {run_name}",
                delete_not_versioned_files=True,
            )
        except Exception:
            _KAGGLE_API.dataset_create_new(str(staging))
        print(
            f"  ↑ best model pushed → dataset {_BEST_DS_SLUG} (ep{epoch}, f1={f1:.4f})"
        )
    except Exception as e:
        print(f"  ⚠ upload best failed: {e}")


print(f"Checkpoint helpers ready. start_epoch = {start_epoch}")

In [ ]:
# ── Training + evaluation functions ─────────────────────────────────────────


def compute_metrics(y_true, y_pred, prefix):
    arr_t, arr_p = np.array(y_true), np.array(y_pred)
    return {
        f"{prefix}_accuracy": float((arr_t == arr_p).mean()),
        f"{prefix}_macro_f1": float(
            f1_score(arr_t, arr_p, average="macro", zero_division=0)
        ),
        f"{prefix}_real_f1": float(
            f1_score(arr_t, arr_p, pos_label=0, average="binary", zero_division=0)
        ),
        f"{prefix}_fake_f1": float(
            f1_score(arr_t, arr_p, pos_label=1, average="binary", zero_division=0)
        ),
        f"{prefix}_real_precision": float(
            precision_score(arr_t, arr_p, pos_label=0, zero_division=0)
        ),
        f"{prefix}_real_recall": float(
            recall_score(arr_t, arr_p, pos_label=0, zero_division=0)
        ),
        f"{prefix}_fake_precision": float(
            precision_score(arr_t, arr_p, pos_label=1, zero_division=0)
        ),
        f"{prefix}_fake_recall": float(
            recall_score(arr_t, arr_p, pos_label=1, zero_division=0)
        ),
    }


def train_one_epoch(epoch):
    model.train()

    shift = CONFIG["training"]["negative_shift"]
    min_b = CONFIG["training"]["min_batch_size"]
    gc_val = CONFIG["training"]["grad_clip"]

    total_class, total_domain = 0.0, 0.0
    all_preds, all_labels = [], []
    n_batches, n_skipped = 0, 0

    pbar = tqdm(loaders["train"], desc=f"Ep {epoch:02d} [train]", leave=False)
    for caption, image, _ in pbar:
        if caption.size(0) < min_b:
            n_skipped += 1
            continue

        caption = caption.to(DEVICE)
        image = image.to(DEVICE)

        det_cap, det_img, det_lbl = make_detection_batch(caption, image, shift)
        event_lbl = get_pseudo_event_labels(det_cap).to(DEVICE)

        class_out, domain_out = model(det_cap, det_img, GRL_LAMBDA)
        l_class = loss_ce(class_out, det_lbl)
        l_domain = loss_ce(domain_out, event_lbl)
        loss = l_class + l_domain

        if not torch.isfinite(loss):
            n_skipped += 1
            optimizer.zero_grad()
            continue

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), gc_val)
        optimizer.step()

        total_class += l_class.item()
        total_domain += l_domain.item()
        n_batches += 1

        all_preds.extend(class_out.argmax(1).cpu().numpy())
        all_labels.extend(det_lbl.cpu().numpy())

        pbar.set_postfix(
            cls=f"{l_class.item():.3f}",
            dom=f"{l_domain.item():.3f}",
            skip=n_skipped,
        )

    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

    metrics = compute_metrics(all_labels, all_preds, "train")
    metrics.update(
        {
            "train_loss_class": round(total_class / max(1, n_batches), 4),
            "train_loss_domain": round(total_domain / max(1, n_batches), 4),
            "skipped_batches": n_skipped,
        }
    )
    return metrics


@torch.no_grad()
def evaluate(loader, split_name):
    model.eval()

    shift = CONFIG["training"]["negative_shift"]
    total_loss, n_batches = 0.0, 0
    all_preds, all_labels, all_scores = [], [], []

    for caption, image, _ in tqdm(loader, desc=f"  [{split_name}]", leave=False):
        caption = caption.to(DEVICE)
        image = image.to(DEVICE)

        det_cap, det_img, det_lbl = make_detection_batch(caption, image, shift)
        class_out, _ = model(
            det_cap, det_img, grl_lambda=0.0
        )  # GRL irrelevant, no backward in eval
        l = loss_ce(class_out, det_lbl)

        if torch.isfinite(l):
            total_loss += l.item()
        n_batches += 1

        all_preds.extend(class_out.argmax(1).cpu().numpy())
        all_labels.extend(det_lbl.cpu().numpy())
        all_scores.extend(F.softmax(class_out, dim=1)[:, 1].cpu().numpy())  # P(fake)

    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

    metrics = compute_metrics(all_labels, all_preds, split_name)
    metrics[f"{split_name}_loss"] = round(total_loss / max(1, n_batches), 4)
    try:
        metrics[f"{split_name}_auc_roc"] = float(roc_auc_score(all_labels, all_scores))
    except ValueError:
        metrics[f"{split_name}_auc_roc"] = float("nan")
    metrics["confusion_matrix"] = confusion_matrix(
        all_labels, all_preds, labels=[0, 1]
    ).tolist()
    return metrics


print("Training functions defined.")

In [ ]:
# ── Run Training ─────────────────────────────────────────────────────────────
history = []
best_val_f1 = -1.0
best_val_acc = -1.0
best_epoch = -1
epochs_no_improve = 0
best_ckpt_path = run_dir / "best_macro_f1.pth"
acc_ckpt_path = run_dir / "best_accuracy.pth"
CKPT_EVERY = CONFIG["checkpointing"]["checkpoint_every"]
PATIENCE = CONFIG["training"]["early_stopping_patience"]

print(
    f"Training EANN for {CONFIG['training']['max_epochs']} epochs (early stopping patience={PATIENCE})"
)
print(f"Device: {DEVICE}")
print()

try:
    for epoch in range(start_epoch, CONFIG["training"]["max_epochs"]):
        tr = train_one_epoch(epoch)
        va = evaluate(loaders["dev"], "val")
        scheduler.step()

        rec = {
            "epoch": epoch,
            "train_loss_class": tr["train_loss_class"],
            "train_loss_domain": tr["train_loss_domain"],
            "train_accuracy": tr["train_accuracy"],
            "train_macro_f1": tr["train_macro_f1"],
            "val_loss": va["val_loss"],
            "val_accuracy": va["val_accuracy"],
            "val_macro_f1": va["val_macro_f1"],
            "val_real_f1": va["val_real_f1"],
            "val_fake_f1": va["val_fake_f1"],
            "val_auc_roc": va["val_auc_roc"],
            "lr": optimizer.param_groups[0]["lr"],
            "skipped_batches": tr["skipped_batches"],
        }
        history.append(rec)

        # Periodic checkpoint
        if (epoch + 1) % CKPT_EVERY == 0:
            save_checkpoint(run_dir / f"ckpt_ep{epoch}.pth", epoch, history, rec)

        # Best val macro-F1
        if va["val_macro_f1"] > best_val_f1:
            best_val_f1 = va["val_macro_f1"]
            best_epoch = epoch
            epochs_no_improve = 0
            save_checkpoint(best_ckpt_path, epoch, history, rec)
            kaggle_upload_best(best_ckpt_path, epoch, best_val_f1)
            print(
                f"  ★ best val_f1={best_val_f1:.4f} @ epoch {epoch} → {best_ckpt_path.name}"
            )
        else:
            epochs_no_improve += 1

        # Best val accuracy (secondary)
        if va["val_accuracy"] > best_val_acc:
            best_val_acc = va["val_accuracy"]
            save_checkpoint(acc_ckpt_path, epoch, history, rec)

        with open(run_dir / "history.json", "w") as f:
            json.dump(history, f, indent=2)
        pd.DataFrame(history).to_csv(run_dir / "history.csv", index=False)

        print(
            f"Ep {epoch:02d} | "
            f"cls={rec['train_loss_class']:.3f} dom={rec['train_loss_domain']:.3f} | "
            f"tr_f1={rec['train_macro_f1']:.3f} | "
            f"val_acc={rec['val_accuracy']:.3f} val_f1={rec['val_macro_f1']:.3f} "
            f"(real={rec['val_real_f1']:.3f} fake={rec['val_fake_f1']:.3f}) | "
            f"skip={rec['skipped_batches']} | no_improve={epochs_no_improve}/{PATIENCE}"
        )

        if epochs_no_improve >= PATIENCE:
            print(
                f"\nEarly stopping: no val_macro_f1 improvement for {PATIENCE} epochs "
                f"(best={best_val_f1:.4f} @ epoch {best_epoch})."
            )
            break

except KeyboardInterrupt:
    _int = run_dir / f"interrupted_ep{epoch}.pth"
    save_checkpoint(_int, epoch, history, {})
    print(f"Interrupted → {_int}")
except RuntimeError as e:
    if "out of memory" in str(e).lower():
        gc.collect()
        torch.cuda.empty_cache()
        raise RuntimeError(f"CUDA OOM — lower batch_size or enable smoke_test.\n{e}")
    raise

print(f"\nDone. Best val_macro_f1={best_val_f1:.4f} at epoch {best_epoch}.")

In [ ]:
# ── Training curves ──────────────────────────────────────────────────────────
df = pd.DataFrame(history)

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
fig.suptitle(f"EANN — {run_name}", fontsize=13)

# Detector loss
ax = axes[0, 0]
ax.plot(df.epoch, df.train_loss_class, label="L_CLASS (train)", color="royalblue")
ax.plot(df.epoch, df.val_loss, label="L_CLASS (val)", color="royalblue", linestyle="--")
ax.set_title("Detector Loss")
ax.legend()
ax.set_xlabel("Epoch")

# Adversary (domain) loss
ax = axes[0, 1]
ax.plot(df.epoch, df.train_loss_domain, label="L_DOMAIN (train)", color="darkorange")
ax.set_title("Adversary (Event) Loss")
ax.legend()
ax.set_xlabel("Epoch")

# Accuracy
ax = axes[0, 2]
ax.plot(df.epoch, df.train_accuracy, label="Train", color="steelblue")
ax.plot(df.epoch, df.val_accuracy, label="Val", color="coral")
ax.set_title("Detection Accuracy")
ax.legend()
ax.set_xlabel("Epoch")

# Macro F1
ax = axes[1, 0]
ax.plot(df.epoch, df.train_macro_f1, label="Train macro-F1", color="steelblue")
ax.plot(df.epoch, df.val_macro_f1, label="Val macro-F1", color="coral")
ax.set_title("Macro F1")
ax.legend()
ax.set_xlabel("Epoch")

# Per-class F1
ax = axes[1, 1]
ax.plot(df.epoch, df.val_real_f1, label="Val Real F1 (0)", color="green")
ax.plot(df.epoch, df.val_fake_f1, label="Val Fake F1 (1)", color="red")
ax.set_title("Val Per-Class F1")
ax.legend()
ax.set_xlabel("Epoch")

# AUC-ROC
ax = axes[1, 2]
ax.plot(df.epoch, df.val_auc_roc, label="Val AUC-ROC", color="purple")
ax.set_title("Val AUC-ROC")
ax.legend()
ax.set_xlabel("Epoch")

plt.tight_layout()
fig.savefig(run_dir / "training_curves.png", dpi=120, bbox_inches="tight")
plt.show()
print(f"Saved → {run_dir / 'training_curves.png'}")

In [ ]:
# ── Final evaluation on test set ─────────────────────────────────────────────
print(f"Loading best checkpoint: {best_ckpt_path}")
load_checkpoint(best_ckpt_path)

test_metrics = evaluate(loaders["test"], "test")
cm = np.array(test_metrics.pop("confusion_matrix"))

print("\n── Test Results ────────────────────────────────────────")
for k, v in test_metrics.items():
    if isinstance(v, float):
        print(f"  {k:<28} {v:.4f}")

# Confusion matrix heatmap
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Real (0)", "Fake (1)"],
    yticklabels=["Real (0)", "Fake (1)"],
    ax=ax,
)
ax.set_title(
    f"EANN — Confusion Matrix (test)\nMacro-F1 = {test_metrics['test_macro_f1']:.4f}"
)
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
plt.tight_layout()
fig.savefig(run_dir / "confusion_matrix_test.png", dpi=120, bbox_inches="tight")
plt.show()

# Save final test results
test_metrics["best_epoch"] = best_epoch
test_metrics["run_name"] = run_name
(run_dir / "test_results.json").write_text(json.dumps(test_metrics, indent=2))
print(f"\nTest results saved → {run_dir / 'test_results.json'}")
print(f"Best checkpoint: {best_ckpt_path}")